In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.shape

In [ ]:
df.info()

In [ ]:
df["Churn"].value_counts(normalize=True)

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].isna().sum()

In [ ]:
df[df["TotalCharges"].isna()][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

In [ ]:
df["Churn"] = np.where(df["Churn"] == "Yes", 1, 0)
df.duplicated().sum(), df["Churn"].mean()

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["TotalCharges"].isna().sum()

In [ ]:
df["tenure_group"] = pd.cut(df["tenure"], bins = [-1, 12, 24, 48, 72], labels=["0-12", "13-24", "25-48", "49+"])
df["tenure_group"].value_counts

In [ ]:
df["avg_monthly_spend"] = np.where(df["tenure"] > 0, df["TotalCharges"]/df["tenure"], df["MonthlyCharges"])
df[["tenure", "TotalCharges", "avg_monthly_spend"]].head()

In [ ]:
service_cols = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)
df["num_services"].value_counts().sort_index()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df.groupby("Contract")["Churn"].mean()

In [ ]:
sns.barplot(x=df["Contract"], y=df["Churn"])
plt.title("Churn Rate by Contract Type")
plt.ylabel("Churn Rate")
plt.show()

In [ ]:
sns.barplot(x=df["tenure_group"], y=df["Churn"], order=["0-12", "13-24", "25-48", "49+"])
plt.title("Churn Rate by Tenure Group")
plt.ylabel("Churn Rate")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(x=df["PaymentMethod"], y=df["Churn"])
plt.title("Churn Rate by Payment Method")
plt.ylabel("Churn Rate")
plt.xticks(rotation=20)
plt.show()

Electronic check customers churn about 3x more than automatic payment customers.

In [ ]:
sns.barplot(x=df["InternetService"], y=df["Churn"])
plt.title("Churn Rate by Internet Service")
plt.ylabel("Churn Rate")
plt.show()

In [ ]:
from scipy.stats import chi2_contingency
table = pd.crosstab(df["Contract"], df["Churn"])
table

In [ ]:
chi2, p, dof, expected = chi2_contingency(table)
p

In [ ]:
table2 = pd.crosstab(df["InternetService"], df["Churn"])
chi2, p2, dof, expected = chi2_contingency(table2)
p2


In [ ]:
table3 = pd.crosstab(df["PaymentMethod"], df["Churn"])
chi2, p3, dof, expected = chi2_contingency(table3)
p3

In [ ]:
pd.crosstab(df["PaymentMethod"], df["Contract"], normalize="index")


In [ ]:
df[df["Contract"] == "Month-to-month"].groupby("PaymentMethod")["Churn"].mean()


In [ ]:
from sklearn.preprocessing import StandardScaler
features = df[["tenure", "MonthlyCharges", "num_services"]]
scaled = StandardScaler().fit_transform(features)
scaled[:3]

In [ ]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["segment"] = kmeans.fit_predict(scaled)
df["segment"].value_counts().sort_index()

In [ ]:
df.groupby("segment")[["tenure", "MonthlyCharges", "num_services", "Churn"]].mean().round(2)

In [ ]:
names = {0: "New & Basic", 1: "Loyal Premium", 2: "At-Risk Mid-Tier", 3: "Loyal Budget"}
df["segment_name"] = df["segment"].map(names)
df.groupby("segment_name")["Churn"].agg(["count", "mean", "sum"]).round(2)

In [ ]:
X = pd.get_dummies(df.drop(columns=["customerID", "Churn", "segment", "segment_name"]), drop_first=True)
y = df["Churn"]
X.shape

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, pred))

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)
print(classification_report(y_test, pred2))

In [ ]:
from xgboost import XGBClassifier
ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, scale_pos_weight=ratio, random_state=42)
xgb.fit(X_train, y_train)
pred3 = xgb.predict(X_test)
print(classification_report(y_test, pred3))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

preds = [pred, pred2, pred3]
probs = [model.predict_proba(X_test)[:, 1], model2.predict_proba(X_test)[:, 1], xgb.predict_proba(X_test)[:, 1]]

results = pd.DataFrame({
    "Model": ["Logistic (plain)", "Logistic (balanced)", "XGBoost"],
    "Precision": [precision_score(y_test, p) for p in preds],
    "Recall": [recall_score(y_test, p) for p in preds],
    "F1": [f1_score(y_test, p) for p in preds],
    "ROC-AUC": [roc_auc_score(y_test, pr) for pr in probs],
}).round(3)
results

In [ ]:
import shap
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

In [ ]:
i = 0
contrib = pd.Series(shap_values[i], index=X_test.columns).sort_values(ascending=False)
prob = xgb.predict_proba(X_test.iloc[[i]])[0, 1]
print(f"Churn probability: {prob:.0%}")
contrib.head(3)

In [121]:
import joblib
joblib.dump(xgb, "../src/churn_model.joblib")
joblib.dump(list(X.columns), "../src/model_columns.joblib")

['../src/model_columns.joblib']